In [79]:
from dotenv import load_dotenv

import os
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
#Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
#Langchain Tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"

os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")#Langsmith Tracing


In [80]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://docs.langchain.com/oss/python/deepagents/models")
docs = loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/deepagents/models', 'title': 'Models - Docs by LangChain', 'description': 'Configure model providers and parameters for Deep Agents', 'language': 'en'}, page_content='Models - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what\'s next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationGet startedModelsOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedQuickstartCustomizationModelsComparison with Claude Agent SDKDeep Agents CodeChangelogDeploymentManaged Deep AgentsBETAGoing to productionExecution environmentToolsBackendsPermissionsMultimodali

In [81]:
from typing import final

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
final_documents = text_splitter.split_documents(docs)
final_documents

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/deepagents/models', 'title': 'Models - Docs by LangChain', 'description': 'Configure model providers and parameters for Deep Agents', 'language': 'en'}, page_content="Models - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationGet startedModelsOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedQuickstartCustomizationModelsComparison with Claude Agent SDKDeep Agents CodeChangelogDeploymentManaged Deep AgentsBETAGoing to productionExecution environmentToolsBackendsPermissionsMultimodalit

In [82]:
from langchain_community.embeddings import OllamaEmbeddings
embedding = OllamaEmbeddings(model="nomic-embed-text")

In [83]:
from langchain_community.vectorstores import FAISS

vectorstoreDB = FAISS.from_documents(final_documents,embedding)

In [84]:
vectorstoreDB

In [85]:
#Querying from a vector DB
query = "what is the Deep Agents string would be?"
result = vectorstoreDB.similarity_search(query)
result[0].page_content

'For more information, see the Eval runs.\n\u200bConfigure model parameters\nPass a model string to create_deep_agent in provider:model format, or pass a configured model instance for full control. Under the hood, model strings are resolved via init_chat_model.\nTo configure model-specific parameters, use init_chat_model or instantiate a provider model class directly:\ninit_chat_modelProvider packagefrom langchain.chat_models import init_chat_model\nfrom deepagents import create_deep_agent'

In [86]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b"
)

print(llm)

metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}} profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True} client=<groq.resources.chat.completions.Completions object at 0x158ce1f30> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x158ce28b0> model_name='openai/gpt-oss-120b' model_kwargs={} groq_api_key=SecretStr('**********')


In [87]:
#Retrieval Chain, Document Chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
        Answer the following questions based on the provided context:    
        <context>
        {context}
        </context>

        Question:
        {input}
"""
)

prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n        Answer the following questions based on the provided context:    \n        <context>\n        {context}\n        </context>\n\n        Question:\n        {input}\n'), additional_kwargs={})])

In [88]:
document_chain = create_stuff_documents_chain(llm,prompt)

In [89]:
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n        Answer the following questions based on the provided context:    \n        <context>\n        {context}\n        </context>\n\n        Question:\n        {input}\n'), additional_kwargs={})])
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.0', 'langchain': '1.3.16'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'ima

from langchain_core.documents import Document
document_chain.invoke({
    "input": "what is the Deep Agents string would be?",
    "context": [Document(page_content="The model identifier must match the format expected by the provider. Some providers use simple names like gpt-5.5; others use namespaced IDs or deployment paths like zai-org/GLM-5.2, so the full Deep Agents string would be baseten:zai-org/GLM-5.2. Check the provider’s model catalog or integration docs for the current identifiers.")]
})

#### However, we want the documents to first come from the receiver we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [90]:
#Retriever

retriever = vectorstoreDB.as_retriever()
from langchain_classic.chains import create_retrieval_chain
retrieval_chain = create_retrieval_chain(retriever,document_chain)
retrieval_chain

##document_chain is responsible in giving the context information. So when we are creating the 
##chain along with the document chain we also give to use the retriever


RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x158f00cd0>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n        Answer the following questions based on the provided context:    \n        <context>\n        {context}\n        </context>\n\n 

In [91]:
response = retrieval_chain.invoke({"input": "what is the Deep Agents string would be?"})
response['answer']

'The **Deep\u202fAgents model string** follows the **`provider:model`** pattern.  \nYou specify the provider name, a colon, and then the model name you want to use.\n\n**Examples**\n\n- `google_genai:gemini-3.6-flash`  \n- `google_genai:gemini-3.1-pro-preview`  \n- `openai:gpt-5.5`  \n- `anthropic:claude-opus-4-8`\n\nSo the Deep\u202fAgents string would be something like **`<provider>:<model>`** (e.g., `google_genai:gemini-3.6-flash`).'

In [92]:
document_chain.invoke({
    "input": "What is the Deep Agents string?",
    "context": result
})

'The “Deep\u202fAgents string” is the model identifier you pass to\u202f`create_deep_agent`.  \nIt must be supplied in the **`provider:model`** format (e.g., `google_genai:gemini-3.6-flash`).'